# evaluate: metrics and the results/<run_name>/ report (M4)

Outputs (`docs/contracts.md`): `metrics.json`, `per_square.csv`, `confusion_matrix.png`.
`run_evaluate.ipynb` calls `evaluate_model` for each trained model.

Milestone: M4 (board assembly + evaluation).

**TODO** (see `docs/M4_TODO.md`)
- [ ] `per_square_accuracy`, `full_board_accuracy`, `confusion_matrix_plot`
- [ ] `evaluate_model`: load the 64 crops per frame in square order, fill the per-square
      rows, add per-class / occupancy-only accuracy and repair rate; also write a per-board
      CSV (`session_id, frame_id, fen_true, fen_ours`) for the chesscog comparison
- [ ] sweep `assembly.low_confidence_threshold` (errors caught vs correct squares flagged)
- [ ] write the self-check (plan at the bottom)

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Sequence

import numpy as np

%run ./config.ipynb
%run ./dataset.ipynb
%run ./board_assembly.ipynb

## Metrics
* per-square accuracy: fraction of crops classified correctly
* full-board accuracy: fraction of boards with all 64 squares right (placement field only)

In [ ]:
def per_square_accuracy(y_true: Sequence[str], y_pred: Sequence[str]) -> float:
    """Fraction of square crops whose predicted label equals the true label.

    Length mismatch -> ValueError; empty input -> nan (not 0: "no data" is not "all wrong").
    """
    # TODO: implement
    raise NotImplementedError


def full_board_accuracy(true_fens: Sequence[str], pred_fens: Sequence[str]) -> float:
    """Fraction of boards where the predicted placement matches exactly (all 64 squares).

    Compares only the piece-placement field, so full FENs and board_fens both work.
    """
    # Design: compare fen.split()[0] of each pair; length mismatch -> ValueError; empty -> nan.
    # TODO: implement
    raise NotImplementedError


def confusion_matrix_plot(
    y_true: Sequence[str],
    y_pred: Sequence[str],
    out_path: str | Path,
    normalize: bool = True,
    title: str = "Square classifier confusion matrix",
) -> np.ndarray:
    """Save a 13x13 confusion matrix (rows = true, cols = predicted) and return it."""
    # Design:
    # - sklearn.metrics.confusion_matrix with labels=list(CLASSES) (fixed 13x13 even if some
    #   classes are absent), normalize="true" for row-normalized; np.nan_to_num for empty rows.
    # - ConfusionMatrixDisplay on a matplotlib figure, savefig, plt.close (no leaking figures).
    # TODO: implement
    raise NotImplementedError

## Evaluation loop
One board = one frame (64 crops). The classifier is anything with
`predict_proba(crops) -> (N, 13)` (baseline wrappers or `CNNClassifier`).

In [ ]:
def evaluate_model(cfg, classifier) -> dict:
    """Run `classifier` on the held-out test sessions and write results/<run_name>/.

    cfg: parsed configs/train.yaml (uses data.crops_root, data.test_sessions,
        assembly.low_confidence_threshold, eval.run_name, eval.results_dir).
    classifier: anything with predict_proba(crops) -> (N, 13) (baseline or CNN).
    Returns the metrics dict that is also written to metrics.json.
    """
    # Design:
    # - Accept the namespace or a dict (to_dict). Output dir = results_dir / run_name.
    # - load_manifest on the test sessions; groupby (session_id, frame_id) = one board.
    # - Per board: sort the 64 rows into square-index order, read_crop each, predict_proba,
    #   assemble(probs, threshold); collect true/pred FENs, legal count, and one per-square
    #   row (session_id, frame_id, square, label, pred, confidence, correct, low_confidence).
    # - Write per_square.csv, boards.csv (session_id, frame_id, fen_true, fen_ours) for
    #   chesscog_compare, metrics.json (run_name, test_sessions, n_boards, per-square and
    #   full-board accuracy, legal_rate, per-class / occupancy-only accuracy, repair rate),
    #   and confusion_matrix.png.
    # TODO: implement
    raise NotImplementedError

## Self-check
(was the metric half of tests/test_evaluate.py; the chesscog half is in `chesscog_compare.ipynb`)

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - per_square_accuracy(["P","empty","k","q"], ["P","empty","k","Q"]) == 0.75; empty -> nan;
#   different lengths -> ValueError.
# - full_board_accuracy([STARTING_FEN]*2, [STARTING_BOARD_FEN, board after 1. e4]) == 0.5.
# - confusion_matrix_plot(["P","P","empty"], ["P","p","empty"], tmp/"cm.png") writes the file
#   and returns a (13, 13) array.
# - evaluate_model on a small fixture session (fake manifest + blank crop PNGs, a dummy
#   classifier returning one-hot probs of the true labels) writes metrics.json,
#   per_square.csv and confusion_matrix.png with per_square_accuracy == 1.0.
print("evaluate: self-check not written yet (see plan above)")